In [3]:
import json
import glob
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd

DATA_DIR = Path("/home/chupchik/Рабочий стол/fisrt_stepD/AIT Alert Data Set/8263181/ait_ads")
OUT_DIR = Path("/home/chupchik/Рабочий стол/fisrt_stepD/output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)


DATA_DIR: /home/chupchik/Рабочий стол/fisrt_stepD/AIT Alert Data Set/8263181/ait_ads
OUT_DIR: /home/chupchik/Рабочий стол/fisrt_stepD/output


In [4]:
def _get(d: Dict[str, Any], path: str, default=None):
    cur = d
    for p in path.split("."):
        if not isinstance(cur, dict) or p not in cur:
            return default
        cur = cur[p]
    return cur


def parse_jsonl_or_json(fp: Path) -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    with fp.open("r", encoding="utf-8", errors="replace") as f:
        first = f.read(1)
        f.seek(0)

        # JSONL
        if first != "[":
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    if isinstance(obj, dict):
                        records.append(obj)
                except json.JSONDecodeError:
                    continue
            return records

        # JSON list/object
        try:
            data = json.load(f)
            if isinstance(data, list):
                records.extend([x for x in data if isinstance(x, dict)])
            elif isinstance(data, dict):
                records.append(data)
        except json.JSONDecodeError:
            pass

    return records


In [5]:
def load_wazuh_dataset(data_dir: Path) -> pd.DataFrame:
    files = sorted(Path(p) for p in glob.glob(str(data_dir / "*wazuh*.json")))

    print("Found Wazuh files:", len(files))
    print("Files:", [f.name for f in files])

    if not files:
        raise FileNotFoundError("Не нашёл *wazuh*.json в папке ait_ads")

    rows = []
    for fp in files:
        recs = parse_jsonl_or_json(fp)

        for r in recs:
            # главное время у Wazuh — '@timestamp'
            ts = r.get("@timestamp") or _get(r, "timestamp")

            rows.append({
                "file": fp.name,

                "timestamp": ts,
                "location": _get(r, "location"),
                "agent_id": _get(r, "agent.id"),
                "agent_name": _get(r, "agent.name"),
                "agent_ip": _get(r, "agent.ip"),
                "hostname": _get(r, "predecoder.hostname"),
                "program": _get(r, "predecoder.program_name"),

                "full_log": _get(r, "full_log"),

                "rule_id": _get(r, "rule.id"),
                "rule_description": _get(r, "rule.description"),
                "rule_groups": _get(r, "rule.groups"),
                "rule_level": _get(r, "rule.level"),
            })

    df = pd.DataFrame(rows)

    # типы
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
    df["rule_level"] = pd.to_numeric(df["rule_level"], errors="coerce")

    # groups -> строка
    def groups_to_str(x):
        if isinstance(x, list):
            return ",".join(map(str, x))
        return str(x) if x is not None else None

    df["rule_groups_str"] = df["rule_groups"].apply(groups_to_str)

    return df


In [6]:
def basic_report(df: pd.DataFrame) -> None:
    print("\n=== BASIC INFO ===")
    print("rows:", len(df))
    print("columns:", list(df.columns))

    print("\n=== MISSING (%) ===")
    miss = (df.isna().mean() * 100).sort_values(ascending=False)
    print(miss.round(2).to_string())

    print("\n=== timestamp range ===")
    print("min:", df["timestamp"].min())
    print("max:", df["timestamp"].max())

    print("\n=== rule_level distribution (top 30) ===")
    print(df["rule_level"].value_counts(dropna=False).head(30).to_string())

    print("\n=== top rule_groups_str (top 20) ===")
    print(df["rule_groups_str"].value_counts(dropna=False).head(20).to_string())


In [ ]:
df_wazuh = load_wazuh_dataset(DATA_DIR)
basic_report(df_wazuh)

# parquet — основной формат 
parquet_path = OUT_DIR / "ait_ads_wazuh.parquet"
df_wazuh.to_parquet(parquet_path, index=False)

# сохраняем небольшой сэмпл CSV (например 50k строк)
sample_csv_path = OUT_DIR / "ait_ads_wazuh_sample_50k.csv"
df_wazuh.sample(n=min(50_000, len(df_wazuh)), random_state=42).to_csv(sample_csv_path, index=False)

print("\nSaved:")
print(" -", parquet_path)
print(" -", sample_csv_path)

df_wazuh.head(5)


Found Wazuh files: 8
Files: ['fox_wazuh.json', 'harrison_wazuh.json', 'russellmitchell_wazuh.json', 'santos_wazuh.json', 'shaw_wazuh.json', 'wardbeck_wazuh.json', 'wheeler_wazuh.json', 'wilson_wazuh.json']

=== BASIC INFO ===
rows: 2600263
columns: ['file', 'timestamp', 'location', 'agent_id', 'agent_name', 'agent_ip', 'hostname', 'program', 'full_log', 'rule_id', 'rule_description', 'rule_groups', 'rule_level', 'rule_groups_str']

=== MISSING (%) ===
program             88.93
hostname            88.93
full_log            11.79
location             0.00
file                 0.00
timestamp            0.00
agent_ip             0.00
agent_name           0.00
agent_id             0.00
rule_id              0.00
rule_description     0.00
rule_groups          0.00
rule_level           0.00
rule_groups_str      0.00

=== timestamp range ===
min: 2022-01-14 00:00:09.728242+00:00
max: 2022-02-08 23:47:24+00:00

=== rule_level distribution (top 30) ===
rule_level
5     1576278
3      592722
6    

,file,timestamp,location,agent_id,agent_name,agent_ip,hostname,program,full_log,rule_id,rule_description,rule_groups,rule_level,rule_groups_str
0,fox_wazuh.json,2022-01-15 02:32:32+00:00,/var/log/syslog,18,wazuh-client,172.17.131.81,mail,freshclam,Jan 15 02:32:32 mail freshclam[29266]: Sat Jan...,52507,ClamAV database update,"[clamd, freshclam, virus]",3,"clamd,freshclam,virus"
1,fox_wazuh.json,2022-01-15 02:32:32+00:00,/var/log/syslog,6,wazuh-client,192.168.128.170,taylorcruz-mail,freshclam,Jan 15 02:32:32 taylorcruz-mail freshclam[2851...,52507,ClamAV database update,"[clamd, freshclam, virus]",3,"clamd,freshclam,virus"
2,fox_wazuh.json,2022-01-15 02:32:37+00:00,/var/log/syslog,18,wazuh-client,172.17.131.81,mail,freshclam,Jan 15 02:32:37 mail freshclam[29266]: Sat Jan...,52507,ClamAV database update,"[clamd, freshclam, virus]",3,"clamd,freshclam,virus"
3,fox_wazuh.json,2022-01-15 02:32:42+00:00,/var/log/syslog,18,wazuh-client,172.17.131.81,mail,freshclam,Jan 15 02:32:42 mail freshclam[29266]: Sat Jan...,52507,ClamAV database update,"[clamd, freshclam, virus]",3,"clamd,freshclam,virus"
4,fox_wazuh.json,2022-01-15 02:32:47+00:00,/var/log/syslog,18,wazuh-client,172.17.131.81,mail,freshclam,Jan 15 02:32:47 mail freshclam[29266]: Sat Jan...,52507,ClamAV database update,"[clamd, freshclam, virus]",3,"clamd,freshclam,virus"


In [1]:
import json
from pathlib import Path

sample_file = Path("/home/chupchik/Рабочий стол/fisrt_stepD/AIT Alert Data Set/8263181/ait_ads/fox_aminer.json")

with open(sample_file, "r", encoding="utf-8", errors="replace") as f:
    first_line = f.readline()

print(first_line[:1000])


{"AnalysisComponent": {"AnalysisComponentIdentifier": 3, "AnalysisComponentType": "NewMatchPathDetector", "AnalysisComponentName": "AMiner: New event type.", "Message": "New path(es) detected", "PersistenceFileName": "nmpd", "TrainingMode": true, "AffectedLogAtomPaths": ["/model/type_str", "/model/type/user_acct", "/model/type/user_acct/msg1_str", "/model/type/user_acct/audit_str", "/model/type/user_acct/time", "/model/type/user_acct/colon_str", "/model/type/user_acct/id", "/model/type/user_acct/pid_str", "/model/type/user_acct/pid", "/model/type/user_acct/uid_str", "/model/type/user_acct/uid", "/model/type/user_acct/auid_str", "/model/type/user_acct/auid", "/model/type/user_acct/ses_str", "/model/type/user_acct/ses", "/model/type/user_acct/msg2_str", "/model/type/user_acct/msg2", "/model/type/user_acct/fm/acct", "/model/type/user_acct/opt", "/model/type/user_acct/terminal_str", "/model/type/user_acct/terminal", "/model/type/user_acct/res_str", "/model/type/user_acct/res", "/model/type

In [ ]:
#пытаюсь понять где хранятся метки 

In [2]:
from pathlib import Path

sample_file = Path("/home/chupchik/Рабочий стол/fisrt_stepD/AIT Alert Data Set/8263181/ait_ads/fox_wazuh.json")

with open(sample_file, "r", encoding="utf-8", errors="replace") as f:
    first_line = f.readline()

print(first_line[:2000])


{"predecoder": {"hostname": "mail", "program_name": "freshclam", "timestamp": "Jan 15 02:32:32"}, "agent": {"ip": "172.17.131.81", "name": "wazuh-client", "id": "18"}, "manager": {"name": "wazuh.manager"}, "rule": {"firedtimes": 1, "mail": false, "level": 3, "pci_dss": ["5.2"], "tsc": ["A1.2"], "description": "ClamAV database update", "groups": ["clamd", "freshclam", "virus"], "id": "52507", "nist_800_53": ["SI.3"], "gpg13": ["4.4"], "gdpr": ["IV_35.7.d"]}, "decoder": {"name": "freshclam"}, "full_log": "Jan 15 02:32:32 mail freshclam[29266]: Sat Jan 15 02:32:32 2022 -> ClamAV update process started at Sat Jan 15 02:32:32 2022", "input": {"type": "log"}, "@timestamp": "2022-01-15T02:32:32.000000Z", "location": "/var/log/syslog", "id": "1686147193.86593"}



In [8]:
print("Rows:", len(df_wazuh))
print("Timestamp min:", df_wazuh["timestamp"].min())
print("Timestamp max:", df_wazuh["timestamp"].max())

# Сколько уникальных значений
print("\nUnique rule_id:", df_wazuh["rule_id"].nunique(dropna=True))
print("Unique groups:", df_wazuh["rule_groups_str"].nunique(dropna=True))

# Топ-10 групп
display(df_wazuh["rule_groups_str"].value_counts().head(10))

# Топ-10 описаний правил
display(df_wazuh["rule_description"].value_counts().head(10))

# Распределение уровней
display(df_wazuh["rule_level"].value_counts().sort_index())


Rows: 2600263
Timestamp min: 2022-01-14 00:00:09.728242+00:00
Timestamp max: 2022-02-08 23:47:24+00:00

Unique rule_id: 31
Unique groups: 22


rule_groups_str
web,accesslog,attack                           1572043
ids,suricata                                    306635
ids                                             306097
dovecot,authentication_success                  276855
web,accesslog,web_scan,recon                    121794
clamd,freshclam,virus                             8407
apache,web,access_denied                          2691
web,appsec,attack                                 2287
dovecot,invalid_login,authentication_failed       1103
pam,syslog,authentication_failed                   678
Name: count, dtype: int64

rule_description
Web server 400 error code.                                                                            1571313
IDS event.                                                                                             295032
Dovecot Authentication Success.                                                                        276855
Multiple web server 400 error codes from same source ip.                                               121794
Suricata: Alert - SURICATA TLS invalid record/traffic                                                  105043
Suricata: Alert - SURICATA TLS invalid handshake message                                               105016
Suricata: Alert - ET INFO Observed DNS Query to .biz TLD                                                91761
ClamAV database update                                                                                   8407
Multiple IDS alerts for same id.                                                                       

rule_level
3      592722
4          13
5     1576278
6      297682
8         629
10     131272
11       1667
Name: count, dtype: int64

In [9]:
import numpy as np

def main_group(groups_str: str):
    if not isinstance(groups_str, str) or not groups_str.strip():
        return None
    return groups_str.split(",")[0].strip()

df = df_wazuh.copy()

# y_type: главная группа
df["y_type"] = df["rule_groups_str"].apply(main_group)

# y_priority: маппинг уровня
def map_priority(level):
    if pd.isna(level):
        return None
    level = float(level)
    if level <= 3:
        return "low"
    elif level <= 6:
        return "medium"
    elif level <= 10:
        return "high"
    else:
        return "critical"

df["y_priority"] = df["rule_level"].apply(map_priority)

print(df[["rule_level", "y_priority", "rule_groups_str", "y_type"]].head(10))


   rule_level y_priority        rule_groups_str y_type
0           3        low  clamd,freshclam,virus  clamd
1           3        low  clamd,freshclam,virus  clamd
2           3        low  clamd,freshclam,virus  clamd
3           3        low  clamd,freshclam,virus  clamd
4           3        low  clamd,freshclam,virus  clamd
5           3        low  clamd,freshclam,virus  clamd
6           3        low  clamd,freshclam,virus  clamd
7           3        low  clamd,freshclam,virus  clamd
8           3        low  clamd,freshclam,virus  clamd
9           3        low  clamd,freshclam,virus  clamd


In [ ]:
# оставим только строки, где есть текст лога и есть целевые метки
df_ml = df.dropna(subset=["full_log", "y_type", "y_priority"]).copy()

print("After dropna:", len(df_ml))

# ограничим редкие типы
MIN_COUNT = 2000  # порог 
type_counts = df_ml["y_type"].value_counts()
common_types = set(type_counts[type_counts >= MIN_COUNT].index)

df_ml["y_type"] = df_ml["y_type"].where(df_ml["y_type"].isin(common_types), "other")

print("\nType distribution (after merge rare -> other):")
display(df_ml["y_type"].value_counts().head(20))

print("\nPriority distribution:")
display(df_ml["y_priority"].value_counts())

# уменьшим выборку для скорости (можно увеличить позже)
N_SAMPLE = 300_000
df_s = df_ml.sample(n=min(N_SAMPLE, len(df_ml)), random_state=42)

print("\nSample size:", len(df_s))


After dropna: 2293628

Type distribution (after merge rare -> other):


y_type
web        1696191
ids         306723
dovecot     277989
clamd         8407
apache        2691
other         1627
Name: count, dtype: int64


Priority distribution:


y_priority
medium      1873973
low          286087
high         131901
critical       1667
Name: count, dtype: int64


Sample size: 300000


In [11]:
print("Unique y_type:", df["y_type"].nunique())
display(df["y_type"].value_counts().head(15))

print("\nPriority distribution:")
display(df["y_priority"].value_counts())


Unique y_type: 8


y_type
web        1696191
ids         613358
dovecot     277989
clamd         8407
apache        2691
pam            956
syslog         559
audit          112
Name: count, dtype: int64


Priority distribution:


y_priority
medium      1873973
low          592722
high         131901
critical       1667
Name: count, dtype: int64

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

X = df_s["full_log"].astype(str)
y = df_s["y_priority"].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

priority_clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1,2),
        min_df=3,
        max_features=200_000
    )),
    ("clf", LogisticRegression(
        max_iter=200,
        n_jobs=-1
    ))
])

priority_clf.fit(X_train, y_train)
pred = priority_clf.predict(X_test)

print(classification_report(y_test, pred))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))


/home/chupchik/Рабочий стол/fisrt_stepD/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/home/chupchik/Рабочий стол/fisrt_stepD/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/chupchik/Рабочий стол/fisrt_stepD/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[

              precision    recall  f1-score   support

    critical       0.00      0.00      0.00        48
        high       0.00      0.00      0.00      3475
         low       1.00      1.00      1.00      7475
      medium       0.93      1.00      0.97     49002

    accuracy                           0.94     60000
   macro avg       0.48      0.50      0.49     60000
weighted avg       0.89      0.94      0.91     60000

Confusion matrix:
 [[    0     0     0    48]
 [    0     0     0  3475]
 [    0     0  7472     3]
 [    0     0     0 49002]]


/home/chupchik/Рабочий стол/fisrt_stepD/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [13]:
X = df_s["full_log"].astype(str)
y = df_s["y_type"].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

type_clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1,2),
        min_df=3,
        max_features=200_000
    )),
    ("clf", LogisticRegression(
        max_iter=200,
        n_jobs=-1
    ))
])

type_clf.fit(X_train, y_train)
pred = type_clf.predict(X_test)

print(classification_report(y_test, pred))


/home/chupchik/Рабочий стол/fisrt_stepD/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


              precision    recall  f1-score   support

      apache       1.00      1.00      1.00        67
       clamd       1.00      1.00      1.00       218
     dovecot       1.00      1.00      1.00      7258
         ids       1.00      1.00      1.00      8009
       other       1.00      0.85      0.92        40
         web       1.00      1.00      1.00     44408

    accuracy                           1.00     60000
   macro avg       1.00      0.97      0.99     60000
weighted avg       1.00      1.00      1.00     60000



In [15]:
#balanced version 
priority_clf_bal = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1,2),
        min_df=3,
        max_features=200_000
    )),
    ("clf", LogisticRegression(
        max_iter=200,
        class_weight="balanced"
    ))
])

priority_clf_bal.fit(X_train, y_train)
pred_bal = priority_clf_bal.predict(X_test)

print(classification_report(y_test, pred_bal))
print("Confusion matrix:\n", confusion_matrix(y_test, pred_bal))


              precision    recall  f1-score   support

      apache       1.00      1.00      1.00        67
       clamd       1.00      1.00      1.00       218
     dovecot       1.00      1.00      1.00      7258
         ids       1.00      1.00      1.00      8009
       other       1.00      1.00      1.00        40
         web       1.00      1.00      1.00     44408

    accuracy                           1.00     60000
   macro avg       1.00      1.00      1.00     60000
weighted avg       1.00      1.00      1.00     60000

Confusion matrix:
 [[   67     0     0     0     0     0]
 [    0   218     0     0     0     0]
 [    0     0  7258     0     0     0]
 [    0     0     0  8009     0     0]
 [    0     0     0     0    40     0]
 [    0     0     0     0     0 44408]]


In [16]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

print("Class weights:")
for c, w in zip(classes, weights):
    print(c, "->", w)


Class weights:
apache -> 149.81273408239701
clamd -> 45.76659038901602
dovecot -> 1.3777425688010196
ids -> 1.2487122654762277
other -> 248.4472049689441
web -> 0.2251846514141596


In [17]:
feature_names = priority_clf_bal.named_steps["tfidf"].get_feature_names_out()
coef = priority_clf_bal.named_steps["clf"].coef_

classes = priority_clf_bal.named_steps["clf"].classes_

for i, class_label in enumerate(classes):
    top10 = np.argsort(coef[i])[-10:]
    print(f"\nTop words for class '{class_label}':")
    for idx in reversed(top10):
        print(feature_names[idx])



Top words for class 'apache':
client
index
error pid
var www
by
error
var
www
www intranet
pid

Top words for class 'clamd':
started at
2022 clamav
clamav
clamav update
freshclam
mail freshclam
update process
process started
started
update

Top words for class 'dovecot':
login
mail dovecot
lip
rip
plain rip
method plain
method
dovecot imap
tls session
imap

Top words for class 'ids':
priority
classification
tcp
priority tcp
traffic
suricata
protocol
protocol command
command decode
classification generic

Top words for class 'other':
horde
pam_unix
auth
uid
sshd
tty
for
intranet server
euid
tty dovecot

Top words for class 'web':
0000
http
404
http 404
mozilla
get
0000 get
compatible
mozilla compatible
windows


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

X = df_s["full_log"].astype(str)
y = df_s["y_priority"].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

priority_clf_bal = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1,2),
        min_df=3,
        max_features=200_000
    )),
    ("clf", LogisticRegression(
        max_iter=300,
        class_weight="balanced"
    ))
])

priority_clf_bal.fit(X_train, y_train)
pred = priority_clf_bal.predict(X_test)

print(classification_report(y_test, pred, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))


              precision    recall  f1-score   support

    critical       0.01      0.31      0.02        48
        high       0.07      0.40      0.12      3475
         low       1.00      1.00      1.00      7475
      medium       0.93      0.60      0.73     49002

    accuracy                           0.64     60000
   macro avg       0.50      0.58      0.47     60000
weighted avg       0.89      0.64      0.73     60000

Confusion matrix:
 [[   15     6     0    27]
 [   70  1393     0  2012]
 [    0     0  7473     2]
 [ 1509 18203    14 29276]]


In [19]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)

print("Priority class weights:")
for c, w in sorted(zip(classes, weights), key=lambda x: x[1], reverse=True):
    print(f"{c:>8} -> {w:.3f}")


Priority class weights:
critical -> 314.136
    high -> 4.316
     low -> 2.007
  medium -> 0.306


In [20]:
import numpy as np

feature_names = priority_clf_bal.named_steps["tfidf"].get_feature_names_out()
coef = priority_clf_bal.named_steps["clf"].coef_
classes = priority_clf_bal.named_steps["clf"].classes_

for i, label in enumerate(classes):
    top_pos = np.argsort(coef[i])[-15:][::-1]
    print(f"\nTop ngrams for PRIORITY='{label}':")
    print(", ".join(feature_names[j] for j in top_pos))



Top ngrams for PRIORITY='critical':
25 59, 47 51, 31 06, 18 26, 50624, 18 12, 55 00, 44840, 06 58, 47 37, 04 43, 32774, 57 42, 54702, 38 43

Top ngrams for PRIORITY='high':
tcp 10, 10 27, 2022 07, 47 00, 39, 22 38, 10 06, 08 31, 44, 04 33, 46 443, 14 43, 404, http 404, http

Top ngrams for PRIORITY='low':
login, mail, session, login login, login user, mpid, imap login, tls session, mail dovecot, lip, imap, dovecot imap, rip, method plain, plain rip

Top ngrams for PRIORITY='medium':
443 192, 02, 52 24, 20 27, 06 10, 11 57, 11, tcp 18, 31 51, tcp 52, 183 254, 17 07, 28 2022, 17 23, 37 08
